In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

In [ ]:
class Tutorial4(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*, move, search, **kwargs):
        while True:
            pos = search(name="sign1")
            if pos is None:
                move(v=0.2)
            else:
                # ####### ここから下に「標識が左右0.2[m]以内なら停止し、そうでないなら一定速度で前進する」プログラムを書こう
                if pos.y <= 0.2 and pos.y >= -0.2:
                    move(v=0.0)
                else:
                    move(v=0.2)
                # ####### ここより上にプログラムを書こう
        # ####### プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Tutorial4())
sim.run()
SimDrawer(sim).show()